## 1. Imports

In [ ]:
from transformers import AutoTokenizer
import numpy as np

## 2. Config — all hyperparameters in one place

In [ ]:
D_MODEL = 256
N_HEADS = 8
HEAD_SIZE = D_MODEL // N_HEADS
D_FF = D_MODEL * 4
N_LAYERS = 4
EPSILON = 1e-8

print(f"d_model={D_MODEL}, n_heads={N_HEADS}, head_size={HEAD_SIZE}, d_ff={D_FF}")

## 3. Tokenizer & Input

In [ ]:
tokenizer = AutoTokenizer.from_pretrained("HuggingFaceTB/SmolLM3-3B")
VOCAB_SIZE = tokenizer.vocab_size

text = "Hello bro"
tokens = tokenizer(text)
input_ids = np.array(tokens["input_ids"])
SEQ_LEN = len(input_ids)

print(f"vocab_size={VOCAB_SIZE}, seq_len={SEQ_LEN}, input_ids={input_ids}")

## 4. Activation Functions

In [ ]:
def softmax(x: np.ndarray) -> np.ndarray:

    x = x - np.max(x, axis=-1, keepdims=True)
    exp_x = np.exp(x)
    return exp_x / np.sum(exp_x, axis=-1, keepdims=True)


def relu(x: np.ndarray) -> np.ndarray:
    return np.maximum(0, x)


test = np.array([1.0, 2.0, 3.0])
assert np.isclose(softmax(test).sum(), 1.0), "Softmax must sum to 1"
print("Activations OK:", softmax(test))

## 5. Embeddings

In [ ]:
class TokenEmbedding:
    def __init__(self, vocab_size: int, d_model: int):
        self.weight = (np.random.randn(vocab_size, d_model) * 0.01).astype(np.float32)

    def forward(self, input_ids: np.ndarray) -> np.ndarray:
        return self.weight[input_ids]


class SinusoidalPositionEncoding:
    def __init__(self, max_seq_len: int, d_model: int):
        self.encoding = self._build(max_seq_len, d_model)

    def _build(self, max_seq_len: int, d: int) -> np.ndarray:
        position = np.arange(max_seq_len).reshape(-1, 1)
        div_term = np.exp(np.arange(0, d, 2) * -(np.log(10000.0) / d))
        pos_enc = np.zeros((max_seq_len, d))
        pos_enc[:, 0::2] = np.sin(position * div_term)
        pos_enc[:, 1::2] = np.cos(position * div_term)
        return pos_enc.astype(np.float32)

    def forward(self, seq_len: int) -> np.ndarray:
        return self.encoding[:seq_len]


tok_emb = TokenEmbedding(VOCAB_SIZE, D_MODEL)
pos_enc = SinusoidalPositionEncoding(512, D_MODEL)

x = tok_emb.forward(input_ids) + pos_enc.forward(SEQ_LEN)
print(f"Embedding output shape: {x.shape}")

## 6. Normalization

In [ ]:
class LayerNorm:
    def __init__(self, d_model: int, epsilon: float = 1e-8):
        self.epsilon = epsilon
        self.gamma = np.ones(d_model)
        self.beta = np.zeros(d_model)

    def forward(self, x: np.ndarray) -> np.ndarray:
        mean = np.mean(x, axis=-1, keepdims=True)
        var = np.var(x, axis=-1, keepdims=True)
        x_norm = (x - mean) / np.sqrt(var + self.epsilon)
        return self.gamma * x_norm + self.beta


class RMSNorm:
    def __init__(self, d_model: int, epsilon: float = 1e-8):
        self.epsilon = epsilon
        self.weight = np.ones(d_model)

    def forward(self, x: np.ndarray) -> np.ndarray:
        rms = np.sqrt(np.mean(x**2, axis=-1, keepdims=True) + self.epsilon)
        return (x / rms) * self.weight


ln = LayerNorm(D_MODEL)
rmsn = RMSNorm(D_MODEL)
assert ln.forward(x).shape == x.shape, "LayerNorm shape mismatch"
assert rmsn.forward(x).shape == x.shape, "RMSNorm shape mismatch"
print("Normalization OK — shapes:", ln.forward(x).shape)

## 7. Attention

In [ ]:
class Head:
    def __init__(self, d_model: int, head_size: int, causal_mask: bool = True):
        self.head_size = head_size
        self.causal_mask = causal_mask
        scale = np.sqrt(2.0 / d_model)
        self.W_q = np.random.randn(d_model, head_size).astype(np.float32) * scale
        self.W_k = np.random.randn(d_model, head_size).astype(np.float32) * scale
        self.W_v = np.random.randn(d_model, head_size).astype(np.float32) * scale

    def apply_rope(self, x, cos, sin):

        x_rotated = np.empty_like(x)

        x_rotated[..., 0::2] = -x[..., 1::2]
        x_rotated[..., 1::2] = x[..., 0::2]

        return (x * cos) + (x_rotated * sin)

    def forward(self, x: np.ndarray, cos, sin) -> np.ndarray:
        Q = x @ self.W_q
        K = x @ self.W_k
        V = x @ self.W_v

        Q = self.apply_rope(Q, cos, sin)
        K = self.apply_rope(K, cos, sin)

        """if cos is not None and sin is not None:
            Q = Q @ cos + Q @ sin
            K = K @ cos + K @ sin"""  # Slower approach

        att = (Q @ K.T) / np.sqrt(self.head_size)

        if self.causal_mask:
            seq_len = x.shape[0]
            mask = np.tril(np.ones((seq_len, seq_len)))
            att = np.where(mask == 1, att, -np.inf)

        att = softmax(att)
        out = att @ V
        return out


class MultiHeadAttention:
    def __init__(
        self, d_model: int, n_heads: int, head_size: int, causal_mask: bool = True
    ):
        assert d_model == n_heads * head_size, (
            f"d_model ({d_model}) must equal n_heads ({n_heads}) * head_size ({head_size})"
        )
        self.heads = [Head(d_model, head_size, causal_mask) for _ in range(n_heads)]
        self.W_o = np.random.randn(d_model, d_model).astype(np.float32) * 0.02

    def forward(self, x: np.ndarray, cos, sin) -> np.ndarray:
        head_outputs = [head.forward(x, cos, sin) for head in self.heads]
        concat = np.concatenate(head_outputs, axis=-1)
        return concat @ self.W_o


mha = MultiHeadAttention(D_MODEL, N_HEADS, HEAD_SIZE, causal_mask=True)
out = mha.forward(x)
print(f"MultiHeadAttention output shape: {out.shape}")

## 8. Feed Forward Network

In [ ]:
class FeedForwardNetwork:
    def __init__(self, d_model: int, d_ff: int):
        self.W1 = (np.random.randn(d_model, d_ff) * np.sqrt(2.0 / d_model)).astype(
            np.float32
        )
        self.W2 = (np.random.randn(d_ff, d_model) * np.sqrt(2.0 / d_ff)).astype(
            np.float32
        )
        self.b1 = np.zeros(d_ff, dtype=np.float32)
        self.b2 = np.zeros(d_model, dtype=np.float32)

    def forward(self, x: np.ndarray) -> np.ndarray:
        x = x @ self.W1 + self.b1
        x = relu(x)
        x = x @ self.W2 + self.b2
        return x


ffn = FeedForwardNetwork(D_MODEL, D_FF)
assert ffn.forward(x).shape == x.shape, "FFN shape mismatch"
print(f"FFN output shape: {ffn.forward(x).shape}")

## 9. Transformer Block

In [ ]:
class TransformerBlock:
    def __init__(self, d_model: int, n_heads: int, head_size: int, d_ff: int):
        self.mha = MultiHeadAttention(d_model, n_heads, head_size, causal_mask=True)
        self.ffn = FeedForwardNetwork(d_model, d_ff)
        self.norm1 = RMSNorm(d_model)
        self.norm2 = RMSNorm(d_model)

    def forward(self, x: np.ndarray, cos, sin) -> np.ndarray:
        x = x + self.mha.forward(self.norm1.forward(x, cos, sin))
        x = x + self.ffn.forward(self.norm2.forward(x))
        return x


block = TransformerBlock(D_MODEL, N_HEADS, HEAD_SIZE, D_FF)
assert block.forward(x).shape == x.shape, "TransformerBlock shape mismatch"
print(f"TransformerBlock output shape: {block.forward(x).shape}")

## 10. MiniGPT — Full Model

In [ ]:
class MiniGPT:
    def __init__(
        self,
        vocab_size: int,
        d_model: int,
        n_heads: int,
        head_size: int,
        d_ff: int,
        n_layers: int,
        max_seq_len: int,
    ):
        self.token_emb = TokenEmbedding(vocab_size, d_model)
        self.head_size = head_size
        self.pos_enc = SinusoidalPositionEncoding(max_seq_len, d_model)
        self.blocks = [
            TransformerBlock(d_model, n_heads, head_size, d_ff) for _ in range(n_layers)
        ]
        self.norm = RMSNorm(d_model)
        self.lm_head = (np.random.randn(d_model, vocab_size) * 0.02).astype(np.float32)

    def get_freq(self, seq_len: int, theta: int = 10000) -> np.ndarray:
        inv_freq = 1 / (theta ** (np.arange(0, self.head_size, 2)) / self.head_size)

        t = np.arange(seq_len)
        freqs = np.outer(t, inv_freq)

        emb = np.repeat(freqs, 2, axis=-1)

        return np.cos(emb), np.sin(emb)

    def forward(self, input_ids: np.ndarray) -> np.ndarray:
        seq_len = len(input_ids)

        x = self.token_emb.forward(input_ids)
        # x = x + self.pos_enc.forward(seq_len) It is used only if we apply Sinusoidal Positional Encoding

        cos = self.get_freq(seq_len)[0]
        sin = self.get_freq(seq_len)[1]

        for block in self.blocks:
            x = block.forward(x, cos, sin)

        x = self.norm.forward(x)
        logits = x @ self.lm_head
        return logits

    def predict_next_token(
        self, input_ids: np.ndarray, temperature: float = 1.0
    ) -> int:

        logits = self.forward(input_ids)
        last_logits = logits[-1] / temperature
        probs = softmax(last_logits)
        return int(np.argmax(probs))


model = MiniGPT(
    vocab_size=VOCAB_SIZE,
    d_model=D_MODEL,
    n_heads=N_HEADS,
    head_size=HEAD_SIZE,
    d_ff=D_FF,
    n_layers=N_LAYERS,
    max_seq_len=512,
)

logits = model.forward(input_ids)
print(f"Input shape:  {input_ids.shape}")
print(f"Output shape: {logits.shape}")
print(f"Logits range: [{logits.min():.3f}, {logits.max():.3f}]")

## 11. Inference — Next Token Prediction

In [ ]:
next_token_id = model.predict_next_token(input_ids)
next_token_text = tokenizer.decode([next_token_id])

print(f"Input text:      '{text}'")
print(f"Next token id:   {next_token_id}")
print(f"Next token text: '{next_token_text}'")
print()
print(
    "Note: output is random (weights are not trained) — this only tests the forward pass."
)

## 12. What's next — migrate to PyTorch

This notebook proves the **forward pass is correct**. To actually train MiniGPT:

| Step | What to do |
|------|------------|
| 1 | Replace `np.ndarray` with `torch.Tensor` |
| 2 | Inherit all classes from `nn.Module` |
| 3 | Replace `np.random.randn` weights with `nn.Linear` / `nn.Embedding` |
| 4 | Replace `relu` with `F.gelu` or `SwiGLU` |
| 5 | Add `CrossEntropyLoss` + `AdamW` optimizer |
| 6 | Write training loop with W&B logging |
| 7 | Implement RoPE instead of sinusoidal PE |